# Model Selection and Observations

## Goal

Compare 4 classical ML models on the Titanic dataset using 5-fold cross-validation, tune the most promising one with GridSearchCV, and select the final model based on CV AUC — the most reliable metric for generalization.

---

## Results — 5-Fold Cross-Validation on X_train

| Model | Accuracy | AUC |
|---|---|---|
| Logistic Regression | 0.8300 ± 0.0209 | 0.8615 ± 0.0249 |
| Decision Tree | 0.7600 ± 0.0476 | 0.7507 ± 0.0368 |
| Random Forest | 0.8020 ± 0.0248 | 0.8506 ± 0.0263 |
| XGBoost | 0.7865 ± 0.0098 | 0.8457 ± 0.0051 |

Logistic Regression wins on CV with the highest AUC of 0.8615. After feature engineering, the key predictors — Title, Sex, Pclass, FamilySize, HasCabin — are all strong linear signals for survival, which plays directly into Logistic Regression's strengths. When the hard work is done in feature engineering, simpler models often win.

Decision Tree is the weakest model — lowest AUC and highest std of 0.0368, meaning it is both inaccurate and unstable. A single unpruned tree memorizes noise in the training data and fails to generalize. It is included only as a baseline to show why ensembles exist.

XGBoost has the most interesting pattern — lowest std by far (0.0051) but lowest AUC among the top 3 (0.8457). Low std combined with low mean means the model is consistently conservative — not complex enough with default parameters. Default XGBoost with learning_rate=0.3 takes large steps and converges too quickly without carefully exploring the loss surface. This is underfitting, not overfitting — which is exactly why it is worth tuning.

---

## Why Tune XGBoost and not the Others?

XGBoost is the most sensitive model to hyperparameters by design. Its performance is controlled by three interdependent parameters — how many trees, how deep each tree is, and how large each correction step is. With wrong defaults it underfits. With the right combination it can significantly outperform simpler models.

Logistic Regression has very few hyperparameters and is already performing well. Random Forest is robust with defaults on small datasets. XGBoost is the one model where tuning is likely to make a meaningful difference.

---

## XGBoost Hyperparameter Tuning with GridSearchCV

GridSearchCV tries every possible combination of the given parameters, runs 5-fold CV on each, and returns the combination with the best mean AUC — all without touching the val set.

n_estimators: [100, 300, 500]
max_depth: [3, 4, 5]
learning_rate: [0.01, 0.05, 0.1]

→ 3 × 3 × 3 = 27 combinations
→ each combination gets 5-fold CV
→ 27 × 5 = 135 training runs total
→ n_jobs=-1 runs them in parallel across all CPU cores


**Best params found:**

n_estimators = 500
max_depth = 3
learning_rate = 0.01


Low learning_rate=0.01 means each tree contributes only a tiny correction to the prediction. The model takes very small careful steps toward the minimum of the loss function, which requires more trees (n_estimators=500) to fully converge, but produces a more precise and stable result.

max_depth=3 keeps each tree shallow. On a small dataset like Titanic (~900 training samples), deep trees overfit by memorizing individual passengers. Shallow trees capture only the strongest patterns — Sex, Pclass, Title — which are the ones that generalize to unseen data.

**Tuned XGBoost CV AUC: 0.8690** — up from 0.8457 with defaults. A meaningful improvement from hyperparameter tuning alone, without any change to the data or features.

---

## Final Model Selection — Based on CV AUC

| Model | CV AUC |
|---|---|
| Logistic Regression | 0.8615 |
| Decision Tree | 0.7507 |
| Random Forest | 0.8506 |
| XGBoost (default) | 0.8457 |
| **Tuned XGBoost** | **0.8690** |

**Tuned XGBoost wins** with CV AUC of 0.8690 — the highest across all models and configurations.

CV AUC is used as the decision metric rather than the val set score, because CV evaluates each model across 5 different data splits. A single val set is just one fixed subset of 179 passengers — a model can get lucky or unlucky depending on which passengers ended up there. CV across 5 splits gives a much more stable and trustworthy estimate of how the model will perform on truly unseen data.

---

## ROC Curves — All Models Evaluated on X_val

![ROC Curve](../outputs/roc_curve.png)

| Model | AUC on X_val |
|---|---|
| Logistic Regression | 0.892 |
| Decision Tree | 0.796 |
| Random Forest | 0.903 |
| XGBoost (default) | 0.898 |
| Tuned XGBoost | 0.892 |

Random Forest scored 0.903 on the val set — the highest of all models. This does not contradict the CV-based decision. The val set is a single fixed split and Random Forest happened to perform well on this specific 20% by chance. Its CV AUC of 0.8506 across 5 different splits is the more reliable picture — and it is lower than tuned XGBoost's 0.8690. Selecting a model based on a single val score risks picking one that got lucky on that specific subset, which is exactly the problem CV was designed to solve.

The ROC curves are plotted on the val set to visualize how each model separates survivors from deaths on unseen data, not to make the selection decision.

---

## Confusion Matrix — Tuned XGBoost

![Confusion Matrix](../outputs/confusion_matrix.png)
          precision    recall    f1      support

Died (0) 0.83 0.85 0.84 105
Survived (1) 0.78 0.76 0.77 74
accuracy 0.81 179


The model correctly identified 89 out of 105 deaths and 56 out of 74 survivors. It missed 18 real survivors (false negatives) and wrongly flagged 16 deaths as survived (false positives).

The model performs slightly better on deaths than survivors — 105 deaths vs 74 survivors in the validation set means it has seen proportionally more death examples during training and is more confident predicting that class.

Recall for survivors = 0.76 — of all people who actually survived, 76% were correctly identified. For a survival predictor this is the most important metric — missing a real survivor is the more costly error.

---

## Final Summary

**Final model: Tuned XGBoost**
**Selected by: CV AUC = 0.8690**
**Final accuracy on X_val: 0.81**

The key takeaway from this experiment — feature engineering matters more than model choice on small structured datasets. The work done in features.py drove performance across all models. Tuning XGBoost on top of that clean feature set pushed it above the default models and made it the best generalizing pipeline.